# Lab 11: Diffusion by hand

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Nov 18, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

A diffusion model has two halves. The forward half destroys an image with
noise, and it is plain arithmetic. The reverse half is a network that guesses
the noise, and running that guess backwards is how an image appears.

You will build the arithmetic yourself, then drive a network that was already
trained. There is no training run.

1. **The schedule.** How much of the image survives at each step.
2. **The forward process.** Jump to any step in one line.
3. **What the network predicts.** Its guess of the noise, and the clean image
   that guess implies.
4. **One reverse step.** And why it looks familiar.
5. **The knobs.** Step count and seed.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **Everything is seeded, so your numbers should match a classmate's.** A
  mismatch is a bug, or a cell run out of order. Run All from the top.
- **The understanding is assessed on Quiz 12**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent will write the closed form in one shot. It is much less likely to
point out that your reverse step is the same line as your forward step, and
Part 6 asks you why.

## Step 0: Setup

::: {.callout-note}
## No GPU required
The denoiser is about a million parameters and the images are 28 by 28.
Sampling 16 digits with 50 steps is a few seconds on a laptop CPU. A GPU
runtime does no harm.
:::

In [ ]:
import math
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

SEED = 6600
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("LAB11_DATA", "./data")
T = 400                              # number of noise steps the model knows
WEIGHT_URL = ("https://benjaminhoughton.georgetown.domains/deep-learning/"
              "labs/lab-11/ddpm-mnist.pt")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def fetch(url, name, env):
    override = os.environ.get(env)
    if override and Path(override).exists():
        return Path(override)
    dest = Path(name)
    if not dest.exists() or dest.stat().st_size < 1000:
        print(f"downloading {name}")
        urllib.request.urlretrieve(url, dest)
    return dest


mnist = datasets.MNIST(DATA, train=False, download=True,
                       transform=transforms.ToTensor())
# Scale pixels to [-1, 1]. The noise is standard normal, so the ranges match.
images = mnist.data.float().div(127.5).sub(1.0).unsqueeze(1)

print(f"torch {torch.__version__}  device {DEVICE}")
print(f"{len(images):,} test digits, each {tuple(images.shape[1:])}, "
      f"pixels in [{images.min():.0f}, {images.max():.0f}]")

---

# Part 1 · The schedule

A schedule says how much noise to add at each step. It is written as a list of
small numbers $\beta_t$, one per step. What you usually want is the running
product

$$\bar\alpha_t = \prod_{s=1}^{t} (1 - \beta_s)$$

which is the fraction of the original signal's variance that survives to step
$t$. The betas below are the ones the model was trained with.

In [ ]:
def cosine_betas(T, s=0.008):
    """The cosine schedule. Holds on to structure longer than a linear one."""
    t = torch.linspace(0, T, T + 1)
    f = torch.cos(((t / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    abar = f / f[0]
    return (1 - abar[1:] / abar[:-1]).clamp(1e-5, 0.999)


BETAS = cosine_betas(T)
print(f"{len(BETAS)} betas, from {BETAS[0]:.5f} at the start "
      f"to {BETAS[-1]:.3f} at the end")

## 1a · The running product

Return $\bar\alpha_t$ for every step, as a tensor the same length as `betas`.
`torch.cumprod` does a running product.

In [ ]:
def alpha_bar(betas):
    """Running product of (1 - beta). Same length as betas."""
    # TODO: one line with torch.cumprod.
    raise NotImplementedError("alpha_bar")


ABAR = alpha_bar(BETAS)
for t in (0, 100, 200, 300, T - 1):
    print(f"t = {t:>3}   alpha_bar {ABAR[t]:.4f}   "
          f"signal kept {100 * ABAR[t].sqrt():5.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(ABAR.sqrt().numpy(), color="#2980b9", lw=2.5)
ax.set_xlabel("step t")
ax.set_ylabel("signal kept")
ax.set_title("How much of the original survives")
ax.set_ylim(-0.02, 1.02)
fig.tight_layout()

---

# Part 2 · The forward process

Composing all those small Gaussian steps gives one Gaussian, so you never have
to simulate the chain:

$$x_t = \sqrt{\bar\alpha_t}\; x_0 \;+\; \sqrt{1 - \bar\alpha_t}\; \epsilon,
  \qquad \epsilon \sim N(0, I)$$

No network is involved anywhere in this part.

## 2a · Jump to any step

`x0` is a batch of images, shape `(N, 1, 28, 28)`. `t` is a tensor of `N`
step indices. `noise` is the same shape as `x0`. Return $x_t$.

You will need to reshape `ABAR[t]` to `(N, 1, 1, 1)` so it broadcasts over the
pixels.

In [ ]:
def q_sample(x0, t, noise):
    """The closed-form forward process. No model."""
    # TODO: look up ABAR at t, reshape to (N, 1, 1, 1), then apply the formula.
    raise NotImplementedError("q_sample")


torch.manual_seed(SEED)
x0 = images[:1]
noise = torch.randn_like(x0)
steps = [0, 60, 120, 180, 240, 300, T - 1]
frames = [q_sample(x0, torch.tensor([t]), noise) for t in steps]

fig, axes = plt.subplots(1, len(steps), figsize=(12, 2.2))
for ax, t, f in zip(axes, steps, frames):
    ax.imshow(f[0, 0].numpy(), cmap="Greys_r", vmin=-1, vmax=1)
    ax.set_title(f"t = {t}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.tight_layout()

final_corr = torch.corrcoef(torch.stack([frames[-1].flatten(), x0.flatten()]))[0, 1]
print(f"correlation of the last frame with the original: {final_corr:.3f}")

The last frame is uncorrelated with the digit. That is the whole design: by the
end, the forward process has turned every image into the same thing, a draw
from a standard normal. Which means a draw from a standard normal is a fine
place to *start* going backwards.

---

# Part 3 · What the network predicts

The network is a small U-Net, the same encoder-decoder shape as the one in
Week 7, with the step number mixed into every block. Given $x_t$ and $t$, it
returns its guess of the noise $\epsilon$ that was added.

The class below must match the trained weights exactly, so do not edit it.

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout)
        self.n1 = nn.GroupNorm(8, cout)
        self.n2 = nn.GroupNorm(8, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = F.silu(self.n1(self.c1(x)))
        h = h + self.t(temb)[:, :, None, None]      # the step, added per channel
        h = F.silu(self.n2(self.c2(h)))
        return h + self.skip(x)


class UNet(nn.Module):
    def __init__(self, ch=64, tdim=128):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(),
                                  nn.Linear(tdim, tdim))
        self.d1 = Block(1, ch, tdim)
        self.d2 = Block(ch, ch * 2, tdim)
        self.mid = Block(ch * 2, ch * 2, tdim)
        self.u2 = Block(ch * 4, ch, tdim)
        self.u1 = Block(ch * 2, ch, tdim)
        self.out = nn.Conv2d(ch, 1, 3, padding=1)
        self.pool = nn.AvgPool2d(2)

    def temb(self, t):
        half = self.tdim // 2
        freq = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float()[:, None] * freq[None]
        return self.tmlp(torch.cat([a.sin(), a.cos()], dim=1))

    def forward(self, x, t):
        e = self.temb(t)
        h1 = self.d1(x, e)
        h2 = self.d2(self.pool(h1), e)
        m = self.mid(self.pool(h2), e)
        u = F.interpolate(m, size=h2.shape[-1], mode="nearest")
        u = self.u2(torch.cat([u, h2], dim=1), e)
        u = F.interpolate(u, size=h1.shape[-1], mode="nearest")
        u = self.u1(torch.cat([u, h1], dim=1), e)
        return self.out(u)


weights = fetch(WEIGHT_URL, "ddpm-mnist.pt", "LAB11_WEIGHTS")
model = UNet().to(DEVICE)
model.load_state_dict(torch.load(weights, map_location=DEVICE, weights_only=True))
model.eval()
ABAR = ABAR.to(DEVICE)
print(f"loaded {weights.name}: "
      f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

## 3a · How good is the guess

Nothing to fill in. Noise 512 real digits to a middle step, ask the model what
noise was added, and compare its answer with the noise you actually used.

Before any training, the guess would be about as wrong as the noise itself,
which has variance 1.

In [ ]:
torch.manual_seed(SEED)
x0 = images[:512].to(DEVICE)
true_noise = torch.randn_like(x0)
t = torch.full((len(x0),), 200, device=DEVICE, dtype=torch.long)

with torch.no_grad():
    guess = model(q_sample(x0, t, true_noise), t)

mse = F.mse_loss(guess, true_noise).item()
print(f"noise-prediction error at t = 200: {mse:.3f}   (guessing zero: 1.000)")

## 3b · From the guess to a clean image

If you know $x_t$ and you know the noise, the closed form can be run backwards
to solve for $x_0$. Rearrange the forward equation:

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t}\;\hat\epsilon}{\sqrt{\bar\alpha_t}}$$

Write it. `abar_t` is already shaped to broadcast.

In [ ]:
def predict_x0(x_t, eps_hat, abar_t):
    """Solve the forward formula for the clean image, given a noise guess."""
    # TODO: the rearranged closed form above.
    raise NotImplementedError("predict_x0")


torch.manual_seed(SEED)
x0 = images[:6].to(DEVICE)
noise = torch.randn_like(x0)
fig, axes = plt.subplots(3, 6, figsize=(9, 4.8))
for col, step in enumerate([50, 150, 250, 300, 350, 390]):
    t = torch.full((len(x0),), step, device=DEVICE, dtype=torch.long)
    x_t = q_sample(x0, t, noise)
    with torch.no_grad():
        eps_hat = model(x_t, t)
    x0_hat = predict_x0(x_t, eps_hat, ABAR[t].view(-1, 1, 1, 1)).clamp(-1, 1)
    for row, img in enumerate((x0[col], x_t[col], x0_hat[col])):
        axes[row, col].imshow(img[0].cpu().numpy(), cmap="Greys_r", vmin=-1, vmax=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        axes[row, col].grid(False)
    axes[0, col].set_title(f"t = {step}", fontsize=10)
for row, name in enumerate(("real", "noised", "model's guess")):
    axes[row, 0].set_ylabel(name, rotation=0, ha="right", va="center")
fig.tight_layout()

Read the bottom row from left to right. At small $t$ the guess is nearly
perfect, because there is little noise to remove. By $t = 390$ the input is
almost pure noise and the guess is a blurry average digit. One jump from heavy
noise to a clean image does not work. That is why generation takes many small
steps instead of one.

---

# Part 4 · One reverse step

Here is the reverse process, one step at a time. At step $t$:

1. Ask the model for $\hat\epsilon$.
2. Turn that into a guess of the clean image $\hat{x}_0$. You wrote this.
3. Re-noise $\hat{x}_0$ to the **next, smaller** step, using the model's
   $\hat\epsilon$ as the noise.

Step 3 is the only new piece. Write it.

## 4a · Re-noise to the next step

`x0_hat` is the current guess of the clean image, `eps_hat` is the model's
noise, and `abar_next` is $\bar\alpha$ at the next step, already shaped to
broadcast. Return the image at that next step.

Before you write it, look at `q_sample` from Part 2.

In [ ]:
def ddim_step(x0_hat, eps_hat, abar_next):
    """Move from the current step to the next, smaller one."""
    # TODO: the forward closed form, with x0_hat and eps_hat in place of
    #   the real image and the real noise.
    raise NotImplementedError("ddim_step")


@torch.no_grad()
def sample(n=16, steps=50, seed=SEED, keep=()):
    g = torch.Generator(device="cpu").manual_seed(seed)
    x = torch.randn(n, 1, 28, 28, generator=g).to(DEVICE)
    seq = torch.linspace(T - 1, 0, steps).long().tolist()
    snaps = {}
    for i, t in enumerate(seq):
        tt = torch.full((n,), t, device=DEVICE, dtype=torch.long)
        eps_hat = model(x, tt)
        a_t = ABAR[t].view(1, 1, 1, 1)
        x0_hat = predict_x0(x, eps_hat, a_t).clamp(-1, 1)
        if i + 1 < len(seq):
            x = ddim_step(x0_hat, eps_hat, ABAR[seq[i + 1]].view(1, 1, 1, 1))
        else:
            x = x0_hat
        if i in keep:
            snaps[i] = x.cpu().clone()
    return x.cpu(), snaps


out, snaps = sample(n=8, steps=50, keep={0, 10, 20, 28, 36, 44, 49})
print(f"sampled {len(out)} images, background mean {out.mean():.2f} "
      f"(MNIST background is -1)")

In [ ]:
order = sorted(snaps)
fig, axes = plt.subplots(4, len(order), figsize=(1.3 * len(order), 5.4))
for c, i in enumerate(order):
    for r in range(4):
        axes[r, c].imshow(snaps[i][r, 0].numpy(), cmap="Greys_r", vmin=-1, vmax=1)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
    axes[0, c].set_title(f"step {i + 1}", fontsize=10)
fig.suptitle("From pure noise, 50 steps")
fig.tight_layout()

---

# Part 5 · The knobs

Two settings you will change on any real image model. Nothing to fill in.

## 5a · Step count

Same seed, same model. Only the number of reverse steps changes.

In [ ]:
counts = [3, 5, 10, 25, 50]
fig, axes = plt.subplots(len(counts), 1, figsize=(10, 1.35 * len(counts)))
for ax, c in zip(axes, counts):
    imgs, _ = sample(n=10, steps=c)
    ax.imshow(torch.cat(list(imgs[:, 0]), dim=1).numpy(), cmap="Greys_r",
              vmin=-1, vmax=1)
    ax.set_ylabel(f"{c} steps", rotation=0, ha="right", va="center")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.tight_layout()

## 5b · Seed

Same model, same 25 steps. Only the seed changes. Then the same seed twice.

In [ ]:
a, _ = sample(n=8, steps=25, seed=1)
b, _ = sample(n=8, steps=25, seed=2)
c, _ = sample(n=8, steps=25, seed=1)

fig, axes = plt.subplots(3, 1, figsize=(9, 3.6))
for ax, imgs, name in zip(axes, (a, b, c), ("seed 1", "seed 2", "seed 1 again")):
    ax.imshow(torch.cat(list(imgs[:, 0]), dim=1).numpy(), cmap="Greys_r",
              vmin=-1, vmax=1)
    ax.set_ylabel(name, rotation=0, ha="right", va="center")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.tight_layout()

print(f"seed 1 twice, largest pixel difference: {(a - c).abs().max():.2e}")

---

# Part 6 · What this lab showed

Six questions. If you can answer them, you are ready for the Lab 11 part of
Quiz 12.

1. **Why the forward process needs no network**, and why you never have to run
   its 400 steps one at a time.
2. **What the last forward frame is**, and why that makes pure noise a valid
   place to start generating.
3. **What the network outputs.** Not an image. What, and why that is an easy
   thing to train.
4. **Why one jump from heavy noise to a clean image fails**, using the bottom
   row of Part 3b.
5. **Why `ddim_step` and `q_sample` are the same line of arithmetic**, and
   what is different about what you feed them.
6. **What the step count and the seed each control**, and why logging the
   seed is the only way to reproduce an image you liked.

---

# Part 7 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where
you used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that the dissolving digit, the three-row guess grid, the reverse
   trajectory, and both knob figures are visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Nov 18, 11:59 PM ET**.

The notebook is graded for completion. Part 6 is the study list for the quiz.

::: {.callout-note}
## Run from the top if a number looks off
`sample()` builds its own generator from the seed every time, so rerunning a
sampling cell is safe. Rerunning Part 2 without Part 1 is not, because `ABAR`
would not exist.
:::